# Task #31 (Story #5): Tính nhãn `is_delayed`

Quyết định đã chốt (căn cứ số liệu khảo sát ở Task #30):
- Giữ toàn bộ 99.441 đơn, không lọc bỏ đơn nào.
- Một cột duy nhất `is_delayed` (nullable boolean), 3 trạng thái:
  - `True` — có ngày giao, trễ hơn ngày dự kiến
  - `False` — có ngày giao, đúng hạn
  - `NA` — không có `order_delivered_customer_date` để so sánh (gồm cả 8 đơn status `delivered` bị thiếu ngày, xử lý như các đơn thiếu khác vì không có căn cứ để tính)
- Không thêm cột phụ (`is_delivered`/`delivery_status`) — ai cần biết lý do `NA` thì tra `order_status`/`order_delivered_customer_date` sẵn có trong dataset.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/orders_joined.csv")

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print(df.shape)

(99441, 41)


C:\Users\thanh\AppData\Local\Temp\ipykernel_8368\3275711634.py:3: DtypeWarning: Columns (0: payment_has_boleto, 1: payment_has_credit_card, 2: payment_has_debit_card, 3: payment_has_not_defined, 4: payment_has_voucher) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/orders_joined.csv")


## Tính `is_delayed`

`pandas` nullable boolean dtype (`"boolean"`) cho phép giữ đúng 3 trạng thái `True`/`False`/`pd.NA` trong cùng một cột.

In [2]:
delivered_mask = df["order_delivered_customer_date"].notna()

df["is_delayed"] = pd.array([pd.NA] * len(df), dtype="boolean")
df.loc[delivered_mask, "is_delayed"] = (
    df.loc[delivered_mask, "order_delivered_customer_date"]
    > df.loc[delivered_mask, "order_estimated_delivery_date"]
)

df["is_delayed"].value_counts(dropna=False)

is_delayed
False    88649
True      7827
<NA>      2965
Name: count, dtype: Int64

## Sanity check

Số dòng `NA` phải khớp đúng số đơn thiếu `order_delivered_customer_date` đã đếm ở Task #30 (2.965 đơn).

In [3]:
n_na_label = df["is_delayed"].isna().sum()
n_missing_date = df["order_delivered_customer_date"].isna().sum()
print(f"is_delayed = NA: {n_na_label}")
print(f"order_delivered_customer_date thiếu: {n_missing_date}")
assert n_na_label == n_missing_date, "Số dòng NA của is_delayed phải khớp số đơn thiếu ngày giao"
print("OK — khớp.")

is_delayed = NA: 2965
order_delivered_customer_date thiếu: 2965
OK — khớp.


## Lưu dataset trung gian

Chưa phải bản cuối — bản cuối (sau khi kiểm tra mẫu thủ công) sẽ lưu ở Task #32.

In [4]:
df.to_csv("../data/processed/orders_step3_labeled.csv", index=False)
print("Đã lưu data/processed/orders_step3_labeled.csv:", df.shape)

Đã lưu data/processed/orders_step3_labeled.csv: (99441, 42)
